In [1]:
SID4 = 2733
SEED = 2733
print("SID4 =", SID4)
print("SEED =", SEED)

SID4 = 2733
SEED = 2733


In [2]:
!nvidia-smi
!nvcc --version

Fri Aug 28 19:40:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   53C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
%%writefile vecadd.cu
#include <cstdio>

__global__ void vecAdd(float *A, float *B, float *C, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) {
        C[i] = A[i] + B[i];
    }
}

int main() {
    int n = 10;
    float A[10], B[10], C[10];
    for (int i = 0; i < n; i++) { A[i] = i; B[i] = i * 2; }

    float *dA, *dB, *dC;
    cudaMalloc(&dA, n * sizeof(float));
    cudaMalloc(&dB, n * sizeof(float));
    cudaMalloc(&dC, n * sizeof(float));

    cudaMemcpy(dA, A, n * sizeof(float), cudaMemcpyHostToDevice);
    cudaMemcpy(dB, B, n * sizeof(float), cudaMemcpyHostToDevice);

    vecAdd<<<1, 10>>>(dA, dB, dC, n);

    cudaMemcpy(C, dC, n * sizeof(float), cudaMemcpyDeviceToHost);
    for (int i = 0; i < n; i++) printf("%f + %f = %f\n", A[i], B[i], C[i]);
    return 0;
}

Writing vecadd.cu


In [4]:
!nvcc -arch=sm_75 vecadd.cu -o vecadd
!./vecadd

0.000000 + 0.000000 = 0.000000
1.000000 + 2.000000 = 3.000000
2.000000 + 4.000000 = 6.000000
3.000000 + 6.000000 = 9.000000
4.000000 + 8.000000 = 12.000000
5.000000 + 10.000000 = 15.000000
6.000000 + 12.000000 = 18.000000
7.000000 + 14.000000 = 21.000000
8.000000 + 16.000000 = 24.000000
9.000000 + 18.000000 = 27.000000


In [5]:
%%writefile matmul_test.cu
#include <cstdio>

__global__ void matmulKernel(float *A, float *B, float *C, int N) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    if (row < N && col < N) {
        float sum = 0.0f;
        for (int k = 0; k < N; k++) {
            sum += A[row * N + k] * B[k * N + col];
        }
        C[row * N + col] = sum;
    }
}

int main() {
    int N = 4;
    float A[16], B[16], C[16];
    for (int i = 0; i < 16; i++) { A[i] = 1.0f; B[i] = 2.0f; }

    float *dA, *dB, *dC;
    cudaMalloc(&dA, 16 * sizeof(float));
    cudaMalloc(&dB, 16 * sizeof(float));
    cudaMalloc(&dC, 16 * sizeof(float));
    cudaMemcpy(dA, A, 16 * sizeof(float), cudaMemcpyHostToDevice);
    cudaMemcpy(dB, B, 16 * sizeof(float), cudaMemcpyHostToDevice);

    dim3 threadsPerBlock(4, 4);
    dim3 numBlocks(1, 1);
    matmulKernel<<<numBlocks, threadsPerBlock>>>(dA, dB, dC, N);

    cudaMemcpy(C, dC, 16 * sizeof(float), cudaMemcpyDeviceToHost);
    for (int i = 0; i < 4; i++) {
        for (int j = 0; j < 4; j++) printf("%.1f ", C[i * 4 + j]);
        printf("\n");
    }
    return 0;
}

Writing matmul_test.cu


In [6]:
!nvcc -arch=sm_75 matmul_test.cu -o matmul_test
!./matmul_test

8.0 8.0 8.0 8.0 
8.0 8.0 8.0 8.0 
8.0 8.0 8.0 8.0 
8.0 8.0 8.0 8.0 


In [7]:
%%writefile matmul_v2.cu
#include <cstdio>
#include <cstdlib>
#include <cmath>

#define SEED 2733

__global__ void matmulKernel(float *A, float *B, float *C, int N) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    if (row < N && col < N) {
        float sum = 0.0f;
        for (int k = 0; k < N; k++) {
            sum += A[row * N + k] * B[k * N + col];
        }
        C[row * N + col] = sum;
    }
}

void matmulCPU(float *A, float *B, float *C, int N) {
    for (int i = 0; i < N; i++) {
        for (int j = 0; j < N; j++) {
            float sum = 0.0f;
            for (int k = 0; k < N; k++) {
                sum += A[i * N + k] * B[k * N + j];
            }
            C[i * N + j] = sum;
        }
    }
}

int main() {
    int N = 256;
    int size = N * N;

    float *A = (float*)malloc(size * sizeof(float));
    float *B = (float*)malloc(size * sizeof(float));
    float *C_cpu = (float*)malloc(size * sizeof(float));
    float *C_gpu = (float*)malloc(size * sizeof(float));

    srand(SEED);
    for (int i = 0; i < size; i++) {
        A[i] = (float)rand() / RAND_MAX;
        B[i] = (float)rand() / RAND_MAX;
    }

    matmulCPU(A, B, C_cpu, N);

    float *dA, *dB, *dC;
    cudaMalloc(&dA, size * sizeof(float));
    cudaMalloc(&dB, size * sizeof(float));
    cudaMalloc(&dC, size * sizeof(float));
    cudaMemcpy(dA, A, size * sizeof(float), cudaMemcpyHostToDevice);
    cudaMemcpy(dB, B, size * sizeof(float), cudaMemcpyHostToDevice);

    dim3 threadsPerBlock(16, 16);
    dim3 numBlocks(N / 16, N / 16);
    matmulKernel<<<numBlocks, threadsPerBlock>>>(dA, dB, dC, N);

    cudaMemcpy(C_gpu, dC, size * sizeof(float), cudaMemcpyDeviceToHost);

    float maxDiff = 0.0f;
    for (int i = 0; i < size; i++) {
        float diff = fabs(C_cpu[i] - C_gpu[i]);
        if (diff > maxDiff) maxDiff = diff;
    }
    printf("Biggest difference between CPU and GPU results: %e\n", maxDiff);
    printf("(Should be tiny/near-zero — confirms GPU matches CPU)\n");

    return 0;
}

Writing matmul_v2.cu


In [8]:
!nvcc -arch=sm_75 matmul_v2.cu -o matmul_v2
!./matmul_v2

Biggest difference between CPU and GPU results: 1.907349e-05
(Should be tiny/near-zero — confirms GPU matches CPU)


In [9]:
%%writefile matmul_timed.cu
#include <cstdio>
#include <cstdlib>
#include <cmath>
#include <chrono>

#define SEED 2733

__global__ void matmulKernel(float *A, float *B, float *C, int N) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    if (row < N && col < N) {
        float sum = 0.0f;
        for (int k = 0; k < N; k++) {
            sum += A[row * N + k] * B[k * N + col];
        }
        C[row * N + col] = sum;
    }
}

void matmulCPU(float *A, float *B, float *C, int N) {
    for (int i = 0; i < N; i++)
        for (int j = 0; j < N; j++) {
            float sum = 0.0f;
            for (int k = 0; k < N; k++) sum += A[i * N + k] * B[k * N + j];
            C[i * N + j] = sum;
        }
}

int main() {
    int N = 256;
    int size = N * N;

    float *A = (float*)malloc(size * sizeof(float));
    float *B = (float*)malloc(size * sizeof(float));
    float *C_cpu = (float*)malloc(size * sizeof(float));
    float *C_gpu = (float*)malloc(size * sizeof(float));

    srand(SEED);
    for (int i = 0; i < size; i++) { A[i] = (float)rand()/RAND_MAX; B[i] = (float)rand()/RAND_MAX; }

    // ---- Time the CPU run ----
    auto cpuStart = std::chrono::high_resolution_clock::now();
    matmulCPU(A, B, C_cpu, N);
    auto cpuEnd = std::chrono::high_resolution_clock::now();
    double cpuMs = std::chrono::duration<double, std::milli>(cpuEnd - cpuStart).count();

    // ---- Set up GPU memory ----
    float *dA, *dB, *dC;
    cudaMalloc(&dA, size * sizeof(float));
    cudaMalloc(&dB, size * sizeof(float));
    cudaMalloc(&dC, size * sizeof(float));

    cudaEvent_t t1, t2, t3, t4;
    cudaEventCreate(&t1); cudaEventCreate(&t2); cudaEventCreate(&t3); cudaEventCreate(&t4);

    // ---- Time the H2D transfer ----
    cudaEventRecord(t1);
    cudaMemcpy(dA, A, size * sizeof(float), cudaMemcpyHostToDevice);
    cudaMemcpy(dB, B, size * sizeof(float), cudaMemcpyHostToDevice);
    cudaEventRecord(t2);

    // ---- Time the kernel ----
    dim3 threadsPerBlock(16, 16);
    dim3 numBlocks(N / 16, N / 16);
    cudaEventRecord(t2);
    matmulKernel<<<numBlocks, threadsPerBlock>>>(dA, dB, dC, N);
    cudaEventRecord(t3);

    // ---- Time the D2H transfer ----
    cudaMemcpy(C_gpu, dC, size * sizeof(float), cudaMemcpyDeviceToHost);
    cudaEventRecord(t4);
    cudaEventSynchronize(t4);

    float h2dMs, kernelMs, d2hMs;
    cudaEventElapsedTime(&h2dMs, t1, t2);
    cudaEventElapsedTime(&kernelMs, t2, t3);
    cudaEventElapsedTime(&d2hMs, t3, t4);

    printf("N = %d\n", N);
    printf("CPU time:        %.4f ms\n", cpuMs);
    printf("GPU kernel time: %.4f ms\n", kernelMs);
    printf("H2D+D2H time:    %.4f ms\n", h2dMs + d2hMs);
    printf("End-to-end GPU:  %.4f ms\n", kernelMs + h2dMs + d2hMs);
    printf("Speedup:         %.2fx\n", cpuMs / (kernelMs + h2dMs + d2hMs));

    return 0;
}

Writing matmul_timed.cu


In [10]:
!nvcc -arch=sm_75 matmul_timed.cu -o matmul_timed
!./matmul_timed

N = 256
CPU time:        69.8240 ms
GPU kernel time: 0.1718 ms
H2D+D2H time:    0.4808 ms
End-to-end GPU:  0.6526 ms
Speedup:         106.99x


In [12]:
%%writefile matmul_final.cu
#include <cstdio>
#include <cstdlib>
#include <cmath>
#include <chrono>

#define SEED 2733

__global__ void matmulKernel(float *A, float *B, float *C, int N) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    if (row < N && col < N) {
        float sum = 0.0f;
        for (int k = 0; k < N; k++) {
            sum += A[row * N + k] * B[k * N + col];
        }
        C[row * N + col] = sum;
    }
}

void matmulCPU(float *A, float *B, float *C, int N) {
    for (int i = 0; i < N; i++)
        for (int j = 0; j < N; j++) {
            float sum = 0.0f;
            for (int k = 0; k < N; k++) sum += A[i * N + k] * B[k * N + j];
            C[i * N + j] = sum;
        }
}

void runOneSize(int N) {
    int size = N * N;
    float *A = (float*)malloc(size * sizeof(float));
    float *B = (float*)malloc(size * sizeof(float));
    float *C_cpu = (float*)malloc(size * sizeof(float));
    float *C_gpu = (float*)malloc(size * sizeof(float));

    srand(SEED);
    for (int i = 0; i < size; i++) { A[i] = (float)rand()/RAND_MAX; B[i] = (float)rand()/RAND_MAX; }

    auto cpuStart = std::chrono::high_resolution_clock::now();
    matmulCPU(A, B, C_cpu, N);
    auto cpuEnd = std::chrono::high_resolution_clock::now();
    double cpuMs = std::chrono::duration<double, std::milli>(cpuEnd - cpuStart).count();

    float *dA, *dB, *dC;
    cudaMalloc(&dA, size * sizeof(float));
    cudaMalloc(&dB, size * sizeof(float));
    cudaMalloc(&dC, size * sizeof(float));

    cudaEvent_t t1, t2, t3, t4;
    cudaEventCreate(&t1); cudaEventCreate(&t2); cudaEventCreate(&t3); cudaEventCreate(&t4);

    cudaEventRecord(t1);
    cudaMemcpy(dA, A, size * sizeof(float), cudaMemcpyHostToDevice);
    cudaMemcpy(dB, B, size * sizeof(float), cudaMemcpyHostToDevice);
    cudaEventRecord(t2);

    dim3 threadsPerBlock(16, 16);
    dim3 numBlocks(N / 16, N / 16);
    matmulKernel<<<numBlocks, threadsPerBlock>>>(dA, dB, dC, N);
    cudaEventRecord(t3);

    cudaMemcpy(C_gpu, dC, size * sizeof(float), cudaMemcpyDeviceToHost);
    cudaEventRecord(t4);
    cudaEventSynchronize(t4);

    float h2dMs, kernelMs, d2hMs;
    cudaEventElapsedTime(&h2dMs, t1, t2);
    cudaEventElapsedTime(&kernelMs, t2, t3);
    cudaEventElapsedTime(&d2hMs, t3, t4);

    float maxDiff = 0.0f;
    for (int i = 0; i < size; i++) {
        float diff = fabs(C_cpu[i] - C_gpu[i]);
        if (diff > maxDiff) maxDiff = diff;
    }

    printf("\n=== N = %d ===\n", N);
    printf("CPU time:        %.4f ms\n", cpuMs);
    printf("GPU kernel time: %.4f ms\n", kernelMs);
    printf("H2D+D2H time:    %.4f ms\n", h2dMs + d2hMs);
    printf("End-to-end GPU:  %.4f ms\n", kernelMs + h2dMs + d2hMs);
    printf("Speedup:         %.2fx\n", cpuMs / (kernelMs + h2dMs + d2hMs));
    printf("Max CPU/GPU diff: %e\n", maxDiff);

    free(A); free(B); free(C_cpu); free(C_gpu);
    cudaFree(dA); cudaFree(dB); cudaFree(dC);
}

int main() {
    int sizes[] = {256, 1024, 4096};
    for (int i = 0; i < 3; i++) {
        runOneSize(sizes[i]);
    }
    return 0;
}

Overwriting matmul_final.cu


In [13]:
!nvcc -O3 -arch=sm_75 matmul_final.cu -o matmul_final
!./matmul_final


=== N = 256 ===
CPU time:        20.4408 ms
GPU kernel time: 0.2281 ms
H2D+D2H time:    0.4213 ms
End-to-end GPU:  0.6495 ms
Speedup:         31.47x
Max CPU/GPU diff: 1.907349e-05

=== N = 1024 ===
CPU time:        3429.8649 ms
GPU kernel time: 9.2046 ms
H2D+D2H time:    4.6925 ms
End-to-end GPU:  13.8971 ms
Speedup:         246.80x
Max CPU/GPU diff: 9.155273e-05

=== N = 4096 ===
CPU time:        743422.8464 ms
GPU kernel time: 408.9250 ms
H2D+D2H time:    87.4966 ms
End-to-end GPU:  496.4216 ms
Speedup:         1497.56x
Max CPU/GPU diff: 4.272461e-04


In [15]:
!which nvprof ncu nsys
!ls /usr/local/cuda/bin/ | grep -iE "nsys|ncu|nvprof"

/usr/local/cuda/bin/nvprof
/usr/local/cuda/bin/ncu
ncu
ncu-ui
nvprof


In [16]:
%%writefile matmul_profile.cu
#include <cstdio>
#include <cstdlib>

__global__ void matmulKernel(float *A, float *B, float *C, int N) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    if (row < N && col < N) {
        float sum = 0.0f;
        for (int k = 0; k < N; k++) {
            sum += A[row * N + k] * B[k * N + col];
        }
        C[row * N + col] = sum;
    }
}

int main() {
    int N = 1024;
    int size = N * N;
    float *A = (float*)malloc(size * sizeof(float));
    float *B = (float*)malloc(size * sizeof(float));
    for (int i = 0; i < size; i++) { A[i] = (float)rand()/RAND_MAX; B[i] = (float)rand()/RAND_MAX; }

    float *dA, *dB, *dC;
    cudaMalloc(&dA, size * sizeof(float));
    cudaMalloc(&dB, size * sizeof(float));
    cudaMalloc(&dC, size * sizeof(float));
    cudaMemcpy(dA, A, size * sizeof(float), cudaMemcpyHostToDevice);
    cudaMemcpy(dB, B, size * sizeof(float), cudaMemcpyHostToDevice);

    dim3 threadsPerBlock(16, 16);
    dim3 numBlocks(N / 16, N / 16);
    matmulKernel<<<numBlocks, threadsPerBlock>>>(dA, dB, dC, N);
    cudaDeviceSynchronize();

    float *C = (float*)malloc(size * sizeof(float));
    cudaMemcpy(C, dC, size * sizeof(float), cudaMemcpyDeviceToHost);

    printf("Done.\n");
    return 0;
}

Writing matmul_profile.cu


In [17]:
!nvcc -O3 -arch=sm_75 matmul_profile.cu -o matmul_profile
!nvprof ./matmul_profile

==5408== NVPROF is profiling process 5408, command: ./matmul_profile
Done.
==5408== Profiling application: ./matmul_profile
==5408== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   74.11%  9.1989ms         1  9.1989ms  9.1989ms  9.1989ms  matmulKernel(float*, float*, float*, int)
                   14.02%  1.7401ms         2  870.06us  859.85us  880.27us  [CUDA memcpy HtoD]
                   11.87%  1.4740ms         1  1.4740ms  1.4740ms  1.4740ms  [CUDA memcpy DtoH]
      API calls:   90.97%  179.61ms         3  59.868ms  61.567us  179.48ms  cudaMalloc
                    4.66%  9.2004ms         1  9.2004ms  9.2004ms  9.2004ms  cudaDeviceSynchronize
                    2.49%  4.9189ms         3  1.6396ms  1.0875ms  2.7277ms  cudaMemcpy
                    1.33%  2.6357ms       114  23.119us      89ns  1.4570ms  cuDeviceGetAttribute
                    0.53%  1.0473ms         1  1.0473ms  1.0473ms  1.0473ms  cudaLa